# Laboratorio — Pilas y Colas (Capítulo 6, Goodrich, Tamassia & Goldwasser)

Cuatro ejercicios del capítulo de pilas y colas. Se reutiliza `ArrayStack` y `ArrayQueue` de `goodrich/ch06` como estructuras base — usted solo debe completar la lógica pedida en cada ejercicio.

In [23]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue

## Ejercicio 1 — Vaciar una pila de forma recursiva (con herencia)

*Give a recursive method for removing all the elements from a stack.*

Implemente la clase `StackVaciable`, que **hereda de `ArrayStack`** (la de `goodrich/ch06`) y le agrega un método `vaciar_recursivo()`. Este método debe dejar la pila completamente vacía usando **recursión** (sin ningún `for`/`while`), apoyándose únicamente en los métodos ya heredados de `ArrayStack` (`pop()`, `is_empty()`) — no reimplemente nada del padre, solo agregue el método nuevo.

In [24]:
class StackVaciable(ArrayStack):
    """ArrayStack con un metodo adicional para vaciarse recursivamente."""

    def vaciar_recursivo(self):
        """Remueve recursivamente todos los elementos de la pila (sin bucles)."""
        if len(self)==0:
            return None 
        else:
            self.pop()
            self.vaciar_recursivo()

In [25]:
S1 = StackVaciable()
for x in [5, 3, 7, 9, 1]:
    S1.push(x)

print('len antes de vaciar:', len(S1))
S1.vaciar_recursivo()
print('len despues de vaciar:', len(S1))
print('is_empty:', S1.is_empty())

# Caso borde: pila ya vacía
S2 = StackVaciable()
S2.vaciar_recursivo()
print('pila ya vacia -> is_empty:', S2.is_empty())

len antes de vaciar: 5
len despues de vaciar: 0
is_empty: True
pila ya vacia -> is_empty: True


## Ejercicio 2 — Invertir una pila con `transfer` y dos pilas temporales

*Show how to use the `transfer` function, described in Exercise R-6.3, and two temporary stacks, to replace the contents of a given stack `S` with those same elements, but in reversed order.*

La función `transfer(S, T)` (dada abajo, ya implementada — es el Ejercicio R-6.3) mueve todos los elementos de `S` hacia `T`: el elemento que estaba en la cima de `S` es el primero en insertarse en `T`, y por lo tanto termina en el **fondo** de `T`. Al terminar, `S` queda vacía.

In [26]:
def transfer(S, T):
    """Mueve todos los elementos de S hacia T (S queda vacía).
    El elemento que estaba en la cima de S termina en el fondo de T."""
    while not S.is_empty():
        elemento = S.pop()
        T.push(elemento)


Ahora implemente `invertir_pila(S)`, que debe dejar en `S` los mismos elementos pero en **orden invertido** (lo que antes era la cima queda en el fondo, y viceversa). Solo puede usar `transfer` y dos pilas temporales (`ArrayStack`) — no acceda a `S._data` ni a ninguna estructura auxiliar distinta de pilas.

**Pista:** cada llamada a `transfer` invierte el orden una vez. ¿Cuántas veces hay que invertir, y sobre qué pilas, para que el resultado quede de nuevo en `S` pero invertido?

In [27]:
def invertir_pila(S):
    """Reemplaza el contenido de S por los mismos elementos en orden invertido,
    usando unicamente transfer() y dos pilas temporales."""
    T1 = ArrayStack()
    T2 = ArrayStack()
    transfer(S,T1)
    transfer (T1,T2)
    transfer(T2,S)
    pass

In [28]:
S3 = ArrayStack()
for x in [1, 2, 3, 4, 5]:   # cima queda en 5
    S3.push(x)

invertir_pila(S3)

orden_pop = []
while not S3.is_empty():
    orden_pop.append(S3.pop())

print('orden de salida tras invertir:', orden_pop)
print('esperado                    :', [1, 2, 3, 4, 5])
print('OK' if orden_pop == [1, 2, 3, 4, 5] else 'FALLA')

orden de salida tras invertir: [1, 2, 3, 4, 5]
esperado                    : [1, 2, 3, 4, 5]
OK


## Ejercicio 3 — Evaluación no recursiva de notación postfija

La notación postfija (o polaca inversa) escribe una expresión aritmética sin paréntesis: si `(exp1)op(exp2)` es la versión completamente parentizada de una operación `op`, su versión postfija es `pexp1 pexp2 op`. Por ejemplo, la versión postfija de `((5+2)*(8-3))/4` es `5 2 + 8 3 - * 4 /`.

Implemente `evaluar_postfija(expr)` de forma **no recursiva**, usando una `ArrayStack` como única estructura auxiliar:

- `expr` es un `str` con los tokens separados por espacios (números y los operadores `+ - * /`).
- Recorra los tokens de izquierda a derecha: si el token es un número, apílelo; si es un operador, saque dos operandos de la pila (el segundo sacado es el operando izquierdo), aplique la operación y apile el resultado.
- Al final debe quedar un único valor en la pila: ese es el resultado (retórnelo como `float`).

In [29]:
def evaluar_postfija(expr):
    """Evalua una expresion en notacion postfija (tokens separados por espacios)
    de forma no recursiva, usando una ArrayStack."""
    S = ArrayStack()
    for d in expr.split():
        if d in "+-*/":
            e=S.pop()
            g=S.pop()
            if d == "+":
                resultado= g + e
            elif d == "-":
                resultado = g - e
            elif d == "*":
                resultado = g * e
            else:
                resultado = g / e
            S.push(resultado)
        else:
            S.push(float(d))
    return S.pop()


In [30]:
casos = [
    ('5 2 + 8 3 - * 4 /', 8.75),
    ('3 4 +', 7.0),
    ('10 2 /', 5.0),
    ('2 3 4 * +', 14.0),
    ('7', 7.0),
]
for expr, esperado in casos:
    resultado = evaluar_postfija(expr)
    print(f'{expr!r:22s} -> {resultado}   esperado: {esperado}   {"OK" if resultado == esperado else "FALLA"}')

'5 2 + 8 3 - * 4 /'    -> 8.75   esperado: 8.75   OK
'3 4 +'                -> 7.0   esperado: 7.0   OK
'10 2 /'               -> 5.0   esperado: 5.0   OK
'2 3 4 * +'            -> 14.0   esperado: 14.0   OK
'7'                    -> 7.0   esperado: 7.0   OK


## Ejercicio 4 — Cola implementada con dos pilas

*Describe how to implement the queue ADT using two stacks as instance variables.*

Implemente `QueueDosPilas`, que cumple el ADT de cola (`enqueue`, `dequeue`, `first`, `is_empty`, `__len__`) usando **solo dos `ArrayStack`** como variables de instancia (`_entrada` y `_salida`) — sin listas ni arreglos circulares.

**Idea:** `enqueue` siempre apila en `_entrada`. Cuando se necesite `first`/`dequeue` y `_salida` esté vacía, traslade (con `transfer`) todo el contenido de `_entrada` a `_salida` — eso invierte el orden y deja el elemento más antiguo en la cima de `_salida`. Si `_salida` no está vacía, opere directamente sobre ella.

In [33]:
from goodrich.exceptions import Empty

In [34]:
class QueueDosPilas:
    def __init__(self):
        self._entrada = ArrayStack()
        self._salida = ArrayStack()

    def __len__(self):
        return len(self._entrada)+len(self._salida)

    def is_empty(self):
        return len(self)==0
        pass

    def enqueue(self, e):
        self._entrada.push(e)
        pass

    def _trasvasar_si_hace_falta(self):
        """Si _salida esta vacia, mueve todo el contenido de _entrada a _salida."""
        if self._salida.is_empty():
            transfer(self._entrada,self._salida)
        pass

    def first(self):
        self._trasvasar_si_hace_falta()
        return self._salida.top()
        pass

    def dequeue(self):
        self._trasvasar_si_hace_falta()
        return self._salida.pop()
        pass

In [35]:
Q = QueueDosPilas()
for x in [10, 20, 30]:
    Q.enqueue(x)

print('dequeue ->', Q.dequeue())   # esperado: 10 (FIFO)
Q.enqueue(40)
print('dequeue ->', Q.dequeue())   # esperado: 20
print('dequeue ->', Q.dequeue())   # esperado: 30
print('dequeue ->', Q.dequeue())   # esperado: 40
print('is_empty:', Q.is_empty())

try:
    QueueDosPilas().dequeue()
except Empty as e:
    print('Empty capturado correctamente:', e)

dequeue -> 10
dequeue -> 20
dequeue -> 30
dequeue -> 40
is_empty: True
Empty capturado correctamente: Stack is empty


**Pregunta:** ¿cuál es la complejidad de `dequeue` en el **peor caso** y en el **caso amortizado**, para una secuencia larga de `enqueue`/`dequeue` intercalados? Justifique contando cuántas veces se mueve cada elemento entre las dos pilas durante toda su vida en la cola.

In [36]:
# Respuesta:
# Peor caso de un dequeue individual: O(n) si la salida está vacía y hay n elementos en la entrada, ya que se deben mover todos los elementos de la pila de entrada a la pila de salida.
# Caso amortizado sobre una secuencia de n operaciones: O(1) en promedio, ya que cada elemento se mueve a la pila de salida una sola vez y luego se elimina, lo que significa que el costo total de n operaciones es O(n), resultando en un costo promedio de O(1) por operación.
# Justificacion: Cada elemento se mueve a la pila de salida una sola vez y luego se elimina, lo que significa que el costo total de n operaciones es O(n), resultando en un costo promedio de O(1) por operación.
